In [ ]:
# Merci de modifier les variables lignes 10 à 13
import requests
import xml.etree.ElementTree as ET
import time
import pandas as pd
import re
import html

# VARIABLES
annee_min = 2021 # année début (incluse)
annee_max = 2025 # année fin (incluse)
cible_dblp = "https://dblp.org/db/conf/uss/index.xml" # url DBLP de la conférence, terminée par .xml
nom_fichier = "uss_affiliations.xlsx" # nom du fichier contenant les résultats
cle_api_OpenAlex = "VOTRE_CLE_API" # Obtenir la clé ici : https://openalex.org/settings/api-key


# Cache OpenAlex
cache_openalex = {}

def clean_title(title):
    if not title:
        return ""
    title = re.sub(r"<[^>]+>", "", title)
    title = html.unescape(title)
    return title.strip()

def get_arxiv_id_from_url(arxiv_url):
    if not arxiv_url:
        return None
    match = re.search(r'arxiv\.org/abs/(\d+\.\d+)', arxiv_url, re.IGNORECASE)
    if match:
        return match.group(1)
    match = re.search(r'arxiv\.org/abs/([a-z\-]+/\d+)', arxiv_url, re.IGNORECASE)
    if match:
        return match.group(1)
    return None

def safe_request(url, headers=None, retries=3):
    if headers is None:
        headers = {"User-Agent": f"TQC-extractor/1.0"}
    for i in range(retries):
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code == 200:
                return resp
            elif resp.status_code == 429:
                wait = 5 * (i + 1)
                print(f"⏳ Rate limit... pause {wait}s")
                time.sleep(wait)
            else:
                print(f"⚠️ Erreur HTTP {resp.status_code}: {url}")
                return None
        except Exception as e:
            print(f"⚠️ Erreur requête: {e}")
            time.sleep(2)
    return None

def get_openalex_data(arxiv_id=None, doi=None, title=None):
    headers = {"User-Agent": "TQC-extractor/1.0"}
    work = None
    source = None

    # 🔹 DOI
    if doi:
        doi_clean = doi if doi.startswith("http") else f"https://doi.org/{doi}"
        url = f"https://api.openalex.org/works/{doi_clean}"
        if cle_api_OpenAlex:
            url += f"?api_key={cle_api_OpenAlex}"
        resp = safe_request(url, headers)
        if resp:
            work = resp.json()
            source = "DOI"
        else:
            print(f"⚠️ Échec de la recherche par DOI: {doi}")

    # 🔹 arXiv
    if work is None and arxiv_id:
        url = f"https://api.openalex.org/works?filter=locations.landing_page_url:http://arxiv.org/abs/{arxiv_id}"
        if cle_api_OpenAlex:
            url += f"&api_key={cle_api_OpenAlex}"
        resp = safe_request(url, headers)
        if resp:
            results = resp.json().get("results", [])
            if results:
                work = results[0]
                source = "arXiv"
            else:
                print(f"⚠️ Aucun résultat arXiv trouvé pour: {arxiv_id}")

    # 🔹 titre
    if work is None and title:
        title_encoded = requests.utils.quote(title)
        url = f"https://api.openalex.org/works?filter=title.search:{title_encoded}"
        if cle_api_OpenAlex:
            url += f"&api_key={cle_api_OpenAlex}"
        resp = safe_request(url, headers)
        if resp:
            results = resp.json().get("results", [])
            if results:
                work = results[0]
                source = "titre"
            else:
                print(f"⚠️ Aucun résultat trouvé pour le titre: {title[:50]}...")

    if work is None:
        return {
            "authors": [],
            "title": "",
            "source": None
        }

    auteurs = []
    for a in work.get("authorships", []):
        nom = a.get("author", {}).get("display_name", "")
        insts = [i.get("display_name", "") for i in a.get("institutions", []) if i.get("display_name")]
        auteurs.append({
            "author": nom,
            "affiliations": "; ".join(insts)
        })

    return {
        "authors": auteurs,
        "title": work.get("title", ""),
        "source": source
    }

def extraire(annee_min, annee_max):
    print(f"Analyse : {cible_dblp}")
    rep = safe_request(cible_dblp)
    if not rep:
        print("❌ Échec de la connexion à DBLP")
        return pd.DataFrame()

    try:
        root = ET.fromstring(rep.content)
    except ET.ParseError as e:
        print(f"❌ Erreur de parsing XML: {e}")
        return pd.DataFrame()

    volumes = []
    doi_par_annee = {}  # Déclaration de la variable manquante

    for proc in root.findall('.//proceedings') + root.findall('.//book'):
        titre_volume = proc.findtext('title') or ""
        if "Workshop" in titre_volume:
            continue
        annee = proc.findtext('year')
        url = proc.findtext('url')
        if not annee or not url:
            continue
        try:
            annee = int(annee)
        except:
            continue
        if (annee_min - 1) <= annee <= (annee_max + 1):
            volumes.append((annee, "https://dblp.org/" + url.replace(".html", ".xml")))

    volumes.sort()
    print(f"{len(volumes)} volumes\n")

    rows = []
    for annee_proc, url in volumes:
        print(f"\n=== {annee_proc} ===")
        rep = safe_request(url)
        if not rep:
            print(f"⚠️ Impossible de récupérer {url}")
            continue

        try:
            root = ET.fromstring(rep.content)
        except ET.ParseError as e:
            print(f"❌ Erreur de parsing XML: {e}")
            continue

        for pub in root.findall('.//inproceedings') + root.findall('.//article'):
            titre = clean_title(pub.findtext('title'))
            annee_pub = pub.findtext('year')
            annee_pub = int(annee_pub) if annee_pub else annee_proc

            if not (annee_min <= annee_pub <= annee_max):
                continue

            auteurs_dblp = [a.text for a in pub.findall('author') if a.text]
            arxiv_url = None
            autres = []

            for ee in pub.findall('ee'):
                if ee.text:
                    if "arxiv" in ee.text.lower():
                        arxiv_url = ee.text
                    else:
                        autres.append(ee.text)

            arxiv_id = get_arxiv_id_from_url(arxiv_url)
            doi = autres[0] if autres else None

            if doi:
                doi_par_annee[annee_pub] = doi_par_annee.get(annee_pub, 0) + 1

            # 🔑 CACHE
            key = doi or arxiv_id or titre
            strat = "DOI" if doi else ("arXiv" if arxiv_id else "titre")
            cible_aff = key[:60] + "..." if key and len(key) > 60 else key
            print(f"  [OpenAlex] {strat} → {cible_aff} ...", end=" ")

            if key in cache_openalex:
                result = cache_openalex[key]
                print("⚡ cache")
            else:
                result = get_openalex_data(arxiv_id, doi, titre)
                cache_openalex[key] = result
                print("✅ OK" if result else "❌")
                time.sleep(1)  # rythme API

            if result:
                if not titre:
                    titre = result["title"]
                for a in result["authors"]:
                    rows.append({
                        "annee": annee_pub,
                        "titre": titre,
                        "auteur_dblp": ", ".join(auteurs_dblp),
                        "auteur_openalex": a["author"],
                        "affiliations": a["affiliations"],
                        "doi": doi or "",
                        "source_openalex": result["source"]
                    })
            else:
                for a in auteurs_dblp:
                    rows.append({
                        "annee": annee_pub,
                        "titre": titre,
                        "auteur_dblp": a,
                        "auteur_openalex": "",
                        "affiliations": "",
                        "doi": doi or "",
                        "source_openalex": ""
                    })

        time.sleep(1)

    if not rows:
        print("Aucune donnée à traiter")
        return pd.DataFrame(columns=[
            "annee", "titre", "auteur_dblp", "auteur_openalex",
            "affiliations", "doi", "source_openalex"
        ])

    df = pd.DataFrame(rows)

    print("\n📊 DOI par année :")
    for y in sorted(doi_par_annee):
        print(f"{y}: {doi_par_annee[y]}")

    # Analyse affiliations
    if "affiliations" in df.columns:
        df["affiliations"] = df["affiliations"].fillna("")
        grouped = df.groupby("titre")["affiliations"].apply(
            lambda x: " | ".join([a for a in x if a.strip()])
        )
        nb_total = len(grouped)
        nb_sans = sum(1 for v in grouped if v.strip() == "")
        print("\n📊 Affiliations :")
        print(f"Total titres : {nb_total}")
        print(f"Sans affiliation : {nb_sans}")
        print(f"Taux : {nb_sans / nb_total:.2%}" if nb_total else "")
    else:
        print("\n⚠️ Avertissement : Aucune donnée d'affiliation trouvée")

    return df

# RUN
df = extraire(annee_min, annee_max)
if not df.empty:
    df.to_excel(nom_fichier, index=False)
    print(f"\n💾 Sauvegardé : {nom_fichier}")
else:
    print("\n❌ Aucun résultat à sauvegarder")